# Walkthrough of how to run MRpro inside XNAT

This tutorial shows how to 
 - create a docker image which can run MRpro image reconstruction inside XNAT
 - install the XNAT plugins for MR raw data in XNAT
 - set-up the docker image inside XNAT
 - upload MR raw data to XNAT
 - reconstruct the uploaded data inside XNAT

In [ ]:
import xnat4tests
from pathlib import Path
import os
import subprocess
import time
import zenodo_get

from src.main.mrd_2_xnat import mrd_2_xnat

### 1. Settings
Here we define some versions and the location where the plugins can be found. 

In [ ]:
xnat_version = "1.9.2"
xnat_container_service_version = "3.7.2"

mrd_plugin_link = Path(
    "https://github.com/SyneRBI/xnat-mrd/releases/download/v1.0.0/mrd-plugin-1.0.0.jar"
)

### 2. MRpro Docker image

As a first step we are going to create a docker image with the reconstruction code inside. 
We use the latest MRpro docker image for this: ghcr.io/ptb-mr/mrpro_py313:latest

The easiest to do this is open a terminal and go to `mrpro_xnat/docker`.

Then run `docker build . -t mrpro_for_xnat`. 

This will download the MRpro docker image and create a new image `mrpro_for_xnat`. 
If you want to modify the reconstruction scripts, have a look at `mrpro_xnat/docker/reco_scripts/mr_direct_recon.py`.

Once the docker image is successfully build we can continue with setting up the xnat4tests instance.

### 3. Set up XNAT 4 TESTS
We install xnat4tests as an example of how to interact with XNAT.

In [ ]:
# Create folder for xnat4tests
os.makedirs(Path(os.getcwd()) / ".xnat4tests", exist_ok=True)
xnat_root_dir = Path(os.getcwd()) / ".xnat4tests/root"
docker_build_dir = Path(os.getcwd()) / ".xnat4tests/build"

# Settings for XNAT test server
xnat_config = xnat4tests.Config(
    xnat_root_dir=xnat_root_dir,
    docker_build_dir=docker_build_dir,
    docker_image="mrpro_xnat4tests",
    docker_container="mrpro_xnat4tests",
    build_args={
        "xnat_version": xnat_version,
        "xnat_cs_plugin_version": xnat_container_service_version,
    },
)
xnat4tests.start_xnat(xnat_config)
connection = xnat4tests.connect(xnat_config)

### 4. Download and install plugins

In [ ]:
!curl -L -O {str(mrd_plugin_link)}

In [ ]:
# Check which plugins are installed
plugin_dir = Path("/data/xnat/home/plugins")
status = subprocess.run(
    [
        "docker",
        "exec",
        "mrpro_xnat4tests",
        "ls",
        plugin_dir.as_posix(),
    ],
    check=True,
    capture_output=True,
    text=True,
)
plugins_list = status.stdout.split("\n")

# Install MRD and INTERFILE plugins if not already installed
if mrd_plugin_link.name not in plugins_list:
    try:
        subprocess.run(
            [
                "docker",
                "cp",
                str(mrd_plugin_link.name),
                f"mrpro_xnat4tests:{(plugin_dir / mrd_plugin_link.name).as_posix()}",
            ],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"Command {e.cmd} returned with error code {e.returncode}: {e.output}"
        ) from e

xnat4tests.restart_xnat(xnat_config)
time.sleep(30)  # Wait for XNAT to restart

You can now go to http://localhost:8080 in a browser and login with admin/admin. 
If you go to `Administer` and then `Data types` you will see the MRD data type.

### 5. Setup MRpro reconstruction in XNAT

In [ ]:
!curl -u admin:admin \
  -X POST "http://localhost:8080/xapi/commands" \
  -H "Content-Type: application/json" \
  -d @mr_manifest.json 

### 6. Create project and enable MRpro reconstruction for it

In [ ]:
# Create Project
project_id = "A4IM"
xnat_session = xnat4tests.connect(xnat_config)
response = xnat_session.put(f"/data/archive/projects/{project_id}")
xnat_session.projects.clearcache()

Now we have to enable the commands globally and for the project. Follow the following steps:
 - go to http://localhost:8080 in a browser and login with admin/admin
 - select `Administer` and then `Plugin settings`
 - click on `Command Configurations` and enable "Reconstruct MR data"
 - select `Browse` -> `My Projects` -> `MRpro`
 - click `Project Settings` and enable "Reconstruct MR data"

### 7. Download MR data from zenodo

In [ ]:
data_folder = Path(os.getcwd()) / ".zenodo_data"
os.makedirs(data_folder, exist_ok=True)

# Download MR data
zenodo_get.download(record="17803395", retry_attempts=5, output_dir=data_folder)

### 8. Upload MR raw data

In [ ]:
# Settings
subject_id = "Patient"
mr_session_id = "MrSession"

# Create Subject
project = xnat_session.projects[project_id]
subject = xnat_session.classes.SubjectData(label=subject_id, parent=project)

for mrd_file_path in data_folder.glob("*.mrd"):
    xnat_hdr = mrd_2_xnat(mrd_file_path)
    mr_session = xnat_session.classes.MrSessionData(label=mr_session_id, parent=subject)
    mr_scan_id = str(mrd_file_path.stem).replace(".mrd", "")
    response = xnat_session.put(f"{mr_session.uri}/scans/{mr_scan_id}", query=xnat_hdr)
    mr_session.clearcache()
    scan = mr_session.scans[mr_scan_id]
    scan_resource = scan.create_resource("MR_RAW")
    scan_resource.upload(mrd_file_path, mrd_file_path.name)

### 9. MRpro reconstruction in XNAT

Now we can reconstruct the data:
 - go to http://localhost:8080 in a browser and login with admin/admin
 - select `Browse` -> `My Projects` -> `A4IM`
 - select subject `Patient` -> `MR session`
 - select one of the scans and via `Run Container` select `mr_recon` 
 - once the reconstruction has run through, the image data can be accessed through `Manage Files` -> `DICOM`  

### 10. Upload MR Dicom data

We can of course also upload dicom images directly. 
Depending on the settings of the project, they might first be put into the prearchive. 
You can check the settings in `Browse` -> `My Projects` -> `A4IM` -> `Manage` -> `Define Prearchive Settings`.
If the prearchive is activated then you can check on the progress here `Browse` -> `My Projects` -> `A4IM` -> `View Prearchive`.

In Xnat4Tests the Dicom upload is very slow, so expect a few minutes of waiting before the data is fully uploaded.


In [ ]:
import requests

data_folder = Path(os.getcwd()) / ".zenodo_data"
xnat_url = "http://localhost:8080/"

dicom_zip_file = data_folder / "DICOM.zip"  # Must be zipped

upload_url = (
    f"{xnat_url}/data/services/import"
    f"?project={project_id}&subject={subject_id}&session={mr_session_id}&type=DICOM-zip"
)

response = requests.post(
    upload_url, auth=("admin", "admin"), files={"file": open(str(dicom_zip_file), "rb")}
)

if response.status_code != 200:
    print(f"Error uploading DICOM zip: {response.status_code} - {response.text}")

### Clean-up

In [ ]:
for project in xnat_session.projects:
    for subject in project.subjects.values():
        xnat_session.delete(
            path=f"/data/projects/{project.id}/subjects/{subject.label}",
            query={"removeFiles": "True"},
        )
    project.subjects.clearcache()
xnat_session.disconnect()